# Chapter 3 – Worked examples

This notebook runs the computations of [Chapter 3](../index.md) live: the bridge and the articulation point of
the IEEE 14-bus grid, the Laplacian's zero eigenvalues, effective resistances, Tarjan's DFS table, the coupler
states of a four-busbar station and the Hamming distance ToOp uses as switching distance. The code lives in the
modules next to it (`grid_graph.py`, `tarjan.py`, `station_splits.py`, `hamming.py`); the cells only call them.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import grid_graph as gg
import hamming
import networkx as nx
import numpy as np
import station_splits as ss
import tarjan

## Bridges and articulation points of the IEEE 14-bus grid

The branch list is `ieee14_edges.csv`. A branch is a bridge when its outage islands part of the grid; a bus is
an articulation point when its loss does. Bus 8 hangs off bus 7 through a single transformer.

In [ ]:
edges = gg.load_edges()
graph = gg.build_graph(edges)
print(len(edges), 'branches,', graph.number_of_nodes(), 'buses')
print('bridges and articulation points:', gg.bridges_and_articulation_points(graph))
fig = gg.draw_grid(graph, *gg.bridges_and_articulation_points(graph))

A parallel circuit on the bridge: the multigraph now has two edges between buses 7 and 8, and neither is a
bridge, because removing one leaves the other.

In [ ]:
graph.add_edge(7, 8)
print('bridges with the duplicate:', list(nx.bridges(graph)))
graph.remove_edge(7, 8)

## Zero eigenvalues of the Laplacian count the islands

$L = D - W$ has one zero eigenvalue per connected component (its null space is spanned by the indicator vectors
of the components).

In [ ]:
cut = graph.copy()
cut.remove_edge(7, 8)
print('intact grid :', gg.count_zero_eigenvalues(graph), 'zero eigenvalue(s)')
print('without 7-8 :', gg.count_zero_eigenvalues(cut), 'zero eigenvalue(s)')
print('eigenvalues of the intact grid:', np.round(np.linalg.eigvalsh(gg.laplacian(graph)), 3))

## Effective resistance

With unit resistances, $R_{ij} = (e_i - e_j)^\top L^{+} (e_i - e_j)$. The bridge is the only branch with
$R = 1$: all the current between its ends must flow through it. Every other branch has a parallel path, so
$R < 1$.

In [ ]:
reff = gg.effective_resistances(graph)
print(reff.sort_values('r_eff', ascending=False).head(6).round(4).to_string(index=False))

## Tarjan's DFS on two triangles joined by an edge

Discovery times and low-links; the tree edge $(u, v)$ with $\text{low}[v] > \text{disc}[u]$ is the bridge.

In [ ]:
result = tarjan.tarjan(tarjan.TWO_TRIANGLES)
print('disc:', result['disc'])
print('low :', result['low'])
print('bridges:', result['bridges'], ' articulation points:', result['articulation_points'])

## Coupler states of a ring station

Four busbars joined in a ring; a state is valid when the closed couplers leave exactly two connected
components. Adding a diagonal coupler adds states but no new bipartition.

In [ ]:
for name, couplers in (('ring', ss.RING), ('ring + diagonal', ss.RING_WITH_DIAGONAL)):
    print(name, ss.count_states_and_bipartitions(couplers, ss.N_BUSBARS))
print('unreachable:', [ss.describe(b) for b in ss.unreachable_bipartitions(ss.RING, ss.N_BUSBARS)])

## Hamming distance and the inverted vector

The busbar labels A and B are arbitrary, so a target and its inverse are the same split; ToOp keeps the smaller
of the two distances.

In [ ]:
current = np.array([True, True, False, False, True, False])
target = np.array([False, False, True, True, False, False])
print('plain:', hamming.hamming_distance(current, target), ' inverted:', hamming.hamming_distance(current, ~target))
print('switching distance, inverted?', hamming.switching_distance(current, target))